<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 10 · Dónde nacen los datos: journey map y service blueprint

La semana pasada aprendiste a no confundir una coincidencia con una causa. Hoy la pregunta se mueve un
paso hacia atrás: **antes de analizar un dato hay que saber en qué momento del proceso nació, quién lo
tecleó y qué se perdió por el camino.** Esta es la semana menos técnica del curso y el laboratorio del
miércoles se hace con lápiz, papel y post-its: un customer journey map y un service blueprint del
proceso central del negocio del caso, dibujados a mano y en equipo. Este cuaderno **no** es ese
laboratorio. Es lo que se lleva a la mesa antes de empezar a dibujar: la evidencia numérica de dónde
Comercial Andina captura información, dónde la pierde y qué decisiones está tomando hoy a ciegas.

> **Hoy haces** · Construyes con los datos la evidencia que alimenta el mapa (45 min de cuaderno,
> 45 min de papel). Mides qué proporción de las ventas no se puede atribuir a nadie y en qué canal,
> cuántos clientes cruzan de la tienda al canal en línea, cuánto dura la relación con un cliente y qué
> parte del padrón ni siquiera tiene fecha de alta. Después llenas en equipo las dos plantillas —journey
> map de cinco etapas y service blueprint de cuatro capas— y cierras poniéndole precio a una sola
> mejora del proceso: identificar al cliente en la caja.
>
> **Entrega** · El journey map y el service blueprint del negocio del caso, en papel o en digital, con
> los puntos de captura de datos marcados sobre el blueprint y **al menos tres decisiones que hoy se
> toman sin evidencia** señaladas con un círculo. Más este cuaderno con la cuantificación de una de
> esas tres. Es el insumo directo del catálogo de problemas de la semana 11.
> Nombre de archivo: `lab_10_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Por qué esta semana se trabaja en papel

Las nueve semanas anteriores partían siempre del mismo sitio: un archivo. Y un archivo es el final de
una cadena que empieza mucho antes, con una persona en una caja registradora que decide si pregunta o
no pregunta el nombre del cliente. Esa decisión —tomada en dos segundos, sin supervisión, cuarenta
veces al día— es la que determina si la segmentación de la semana 8 cubre el 92 % del negocio o el
100 %.

El journey map y el service blueprint son las dos herramientas que hacen visible esa cadena. El journey
map se dibuja **desde el cliente**: qué hace, qué siente y dónde se atasca. El blueprint se dibuja
**desde la empresa**: quién lo atiende, qué pasa detrás del mostrador y qué sistema lo sostiene. Puestos
uno encima del otro, cada punto donde una línea cruza la frontera entre el cliente y la empresa es un
punto donde nace un dato o se pierde.

## 2. Dónde nace cada dato de Comercial Andina

Empecemos por el inventario. Las cinco etapas clásicas del journey, y qué archivo de la base guarda
huella de cada una.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas.csv")
ventas_limpias = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv")
productos = pd.read_csv(DATOS / "productos.csv")
sucursales = pd.read_csv(DATOS / "sucursales.csv")
marketing = pd.read_csv(DATOS / "marketing_mensual.csv", parse_dates=["mes"])

referidos = int((clientes["canal_captacion"] == "Referido").sum())
con_historia = ventas_limpias["cliente_id"].nunique()

inventario = pd.DataFrame([
    ("1 · Descubrimiento", "el cliente ve un volante, un anuncio o pasa por la puerta",
     "marketing_mensual.csv", len(marketing),
     "solo el gasto mensual agregado: no se sabe a quién llegó"),
    ("2 · Consideración", "entra, mira, pregunta un precio, compara y decide",
     "— no existe —", 0,
     "cero registros: la visita que no termina en compra no deja rastro"),
    ("3 · Compra", "paga en caja o cierra el pedido en línea",
     "ventas.csv", len(ventas),
     "la etapa mejor cubierta del proceso, con seis niveles de detalle"),
    ("4 · Retención", "vuelve, o deja de volver",
     "se deduce de ventas.csv", con_historia,
     "no hay dato propio: la recencia se calcula, nadie la registra"),
    ("5 · Recomendación", "trae a otro cliente o lo recomienda",
     "clientes.csv · canal_captacion", referidos,
     "se sabe que llegó por referido, no quién lo refirió"),
], columns=["etapa del journey", "qué hace el cliente", "dónde queda registrado",
            "filas disponibles", "qué falta"])

print(inventario.to_string(index=False))
print(f"\ntotal de líneas de venta registradas : {len(ventas):,}")
print(f"total de visitas sin compra registradas: 0")

📌 **De las cinco etapas del journey, una está medida a fondo y las otras cuatro casi no existen.**
La compra deja 80 515 líneas con producto, cantidad, precio, descuento, sucursal y fecha. La
consideración —el cliente que entra, mira y se va— deja exactamente **cero filas**. Y no es un problema
de limpieza que se arregle la semana que viene: ese dato nunca se capturó, así que ninguna consulta lo
va a recuperar.

Esa asimetría explica por qué casi todas las analíticas de retail contestan «qué se vendió» y casi
ninguna contesta «por qué no se vendió». Sobre el blueprint del miércoles, la etapa 2 va marcada en
rojo: **ahí hay una decisión que se toma a ciegas todos los días.**

## 3. Dónde se pierde el dato: el cliente sin nombre

El primer punto de fuga está en la caja. `ventas.csv` es el archivo tal como sale del punto de venta,
antes de la limpieza de la semana 4, y ahí se ve cuántas veces nadie preguntó quién estaba comprando.

In [ ]:
ventas = ventas.merge(sucursales[["sucursal_id", "canal", "ciudad"]], on="sucursal_id",
                      how="left", validate="m:1")
ventas["monto"] = ventas["cantidad"] * ventas["precio_unitario"] * (1 - ventas["descuento"])
ventas["sin_cliente"] = ventas["cliente_id"].isna()

por_canal = ventas.groupby("canal").apply(
    lambda d: pd.Series({
        "líneas": len(d),
        "líneas sin cliente": int(d["sin_cliente"].sum()),
        "% líneas sin cliente": d["sin_cliente"].mean() * 100,
        "facturación": d["monto"].sum(),
        "facturación sin cliente": d.loc[d["sin_cliente"], "monto"].sum(),
        "% facturación sin cliente": d.loc[d["sin_cliente"], "monto"].sum() / d["monto"].sum() * 100,
    }), include_groups=False)

print(por_canal.round(2).to_string(), "\n")
print(f"líneas sin cliente en total : {int(ventas['sin_cliente'].sum()):,} "
      f"({ventas['sin_cliente'].mean():.2%})")
print(f"dinero sin dueño            : {ventas.loc[ventas['sin_cliente'], 'monto'].sum():,.2f} "
      f"({ventas.loc[ventas['sin_cliente'], 'monto'].sum() / ventas['monto'].sum():.2%} "
      f"de la facturación)")
print(f"facturas afectadas          : {ventas.loc[ventas['sin_cliente'], 'factura_id'].nunique():,} "
      f"de {ventas['factura_id'].nunique():,}")

completa = ventas.groupby("factura_id")["sin_cliente"].all()
print(f"facturas totalmente anónimas: {int(completa.sum()):,} ({completa.mean():.2%})")

⚠️ **El dato se pierde igual en los dos canales, y ese es el hallazgo.** En tienda no se identifica
el 7,80 % de las líneas y en línea el 7,77 %: prácticamente lo mismo. En una tienda física eso se
entiende —el cliente tiene prisa, el cajero no pregunta— pero **en el canal en línea el cliente teclea
sus datos para que le llegue el pedido**. Que ahí falte el identificador en uno de cada trece registros
no es un problema de volumen: es un fallo de integración entre la tienda en línea y el sistema de
facturación, y se arregla en el sistema, no con la gente.

En dinero son 212 840,97 dólares —el 7,49 % de la facturación— que existen, se cobraron y no se le
pueden atribuir a nadie. Sobre el blueprint, esa es una flecha que sale de la capa de sistemas y no
llega a ninguna parte.

El segundo punto de fuga es lo contrario: un dato que sí existe y que nadie está mirando.

In [ ]:
ventas_limpias["monto"] = (ventas_limpias["cantidad"] * ventas_limpias["precio_unitario"]
                           * (1 - ventas_limpias["descuento"]))
compras = ventas_limpias[~ventas_limpias["es_devolucion"]].merge(
    sucursales[["sucursal_id", "canal"]], on="sucursal_id", how="left", validate="m:1")

canales_por_cliente = compras.groupby("cliente_id")["canal"].nunique()
perfil_canal = canales_por_cliente.map({1: "un solo canal", 2: "los dos canales"})
solo = compras.groupby("cliente_id")["canal"].agg(lambda s: s.iloc[0])
perfil_canal = np.where(canales_por_cliente == 2, "Tienda + Online", "solo " + solo)

valor = compras.groupby("cliente_id")["monto"].sum()
multicanal = pd.DataFrame({"perfil": perfil_canal, "valor": valor}, index=valor.index)

resumen = (multicanal.groupby("perfil")["valor"]
           .agg(clientes="size", valor_medio="mean", valor_mediano="median", facturación="sum")
           .sort_values("clientes", ascending=False))
resumen["% de clientes"] = resumen["clientes"] / resumen["clientes"].sum() * 100
resumen["% de la facturación"] = resumen["facturación"] / resumen["facturación"].sum() * 100
print(resumen.round(2).to_string())

ambos = resumen.loc["Tienda + Online"]
print(f"\nEl cliente que compra en los dos canales vale "
      f"{ambos['valor_medio'] / resumen.loc['solo Tienda', 'valor_medio']:.1f} veces más "
      f"que el que solo compra en tienda.")

📌 **El 77,61 % de los clientes de Comercial Andina compra en la tienda y en línea, y ese cliente
vale 2,8 veces más** que el que solo va a la tienda: 1 948,08 dólares acumulados contra 688,95. Los 1 359
clientes multicanal se llevan el 92,10 % de la facturación.

Ahora la parte incómoda: **ese número solo se puede calcular porque el cliente está identificado en los
dos sitios.** Para las 5 187 facturas sin identificador, el mismo comprador aparece como dos personas
distintas —una en tienda, una en línea— o como nadie. La empresa que no identifica en caja no es que
tenga un dato menos: es que tiene el hallazgo más importante de su journey invisible.

El tercer bloque de evidencia mide el tiempo, que es la dimensión que el journey map dibuja en el eje
horizontal y que casi ninguna base registra explícitamente.

In [ ]:
vida = compras.groupby("cliente_id")["fecha"].agg(primera="min", ultima="max", facturas="nunique")
vida["días de relación"] = (vida["ultima"] - vida["primera"]).dt.days
HOY = compras["fecha"].max()
vida["días desde la última compra"] = (HOY - vida["ultima"]).dt.days

print(f"fecha de corte: {HOY:%d-%m-%Y}\n")
print(vida[["días de relación", "días desde la última compra"]].describe().round(1).to_string())
una_visita = int((vida["días de relación"] == 0).sum())
print(f"\nclientes con una sola visita en toda su historia : {una_visita:,} "
      f"({una_visita / len(vida):.2%})")
print(f"clientes con más de 180 días sin comprar         : "
      f"{int((vida['días desde la última compra'] > 180).sum()):,} "
      f"({(vida['días desde la última compra'] > 180).mean():.2%})")

sin_alta = clientes["fecha_alta"].isna()
ciudad_sucia = clientes["ciudad"] != clientes["ciudad"].str.strip().str.title().replace(
    {"Guayaquíl": "Guayaquil"})
print(f"\nclientes del padrón sin fecha de alta            : {int(sin_alta.sum()):,} "
      f"({sin_alta.mean():.2%})")
print(f"clientes con la ciudad mal escrita               : {int(ciudad_sucia.sum()):,} "
      f"({ciudad_sucia.mean():.2%})")
print(f"\ncómo dice el padrón que llegó cada cliente:")
print(clientes["canal_captacion"].value_counts().to_string())

La relación media dura 508 días y la mediana 572, casi año y medio: Comercial Andina **sí** tiene
clientes de largo plazo. Pero 160 de ellos —el 9,14 %— compraron una sola vez y no volvieron, y 504 —el
28,78 %— llevan más de medio año sin aparecer. Ninguna de las dos cosas está registrada en ningún sitio:
son cálculos que alguien tiene que hacer. **Nadie en la empresa recibe una alerta cuando un cliente deja
de comprar**, y eso, sobre el blueprint, es una casilla vacía en la capa de sistemas.

Y el padrón tiene sus propios agujeros: 54 clientes (el 3,00 %) no tienen fecha de alta, así que de
ellos no se puede saber cuánto tardaron en hacer la primera compra; y 162 (el 9,00 %) tienen la ciudad
mal escrita, lo que significa que alguien la teclea a mano en un campo libre. Los dos son defectos de
**formulario**, no de análisis: se arreglan en el punto donde el dato nace.

Todo junto, esta es la hoja de evidencia que se lleva a la mesa el miércoles.

In [ ]:
evidencia = pd.DataFrame([
    ("1 · Descubrimiento", "¿de dónde vienen los clientes nuevos?",
     f"{len(marketing)} meses de inversión agregada, sin cliente", "NO"),
    ("2 · Consideración", "¿cuántos entran y no compran?",
     "0 registros de visitas sin compra", "NO"),
    ("3 · Compra", "¿quién compró qué y cuándo?",
     f"{len(ventas):,} líneas · {ventas['sin_cliente'].mean():.2%} sin identificar", "SÍ"),
    ("3 · Compra", "¿cuánto dinero no se le puede atribuir a nadie?",
     f"{ventas.loc[ventas['sin_cliente'], 'monto'].sum():,.2f} "
     f"({ventas.loc[ventas['sin_cliente'], 'monto'].sum() / ventas['monto'].sum():.2%})", "SÍ"),
    ("3 · Compra", "¿el cliente cruza de canal?",
     f"{int(resumen.loc['Tienda + Online', 'clientes']):,} de {len(vida):,} clientes "
     f"({resumen.loc['Tienda + Online', '% de clientes']:.2f} %)", "SÍ"),
    ("4 · Retención", "¿cuánto dura la relación?",
     f"mediana {vida['días de relación'].median():.0f} días", "SÍ"),
    ("4 · Retención", "¿quién dejó de comprar?",
     f"{int((vida['días desde la última compra'] > 180).sum()):,} clientes "
     f"({(vida['días desde la última compra'] > 180).mean():.2%}) · nadie recibe la alerta", "NO"),
    ("4 · Retención", "¿por qué se fue?",
     "el motivo no está en ninguna tabla", "NO"),
    ("5 · Recomendación", "¿quién refirió a quién?",
     f"{referidos} clientes marcados como referidos, sin decir por quién", "NO"),
], columns=["etapa", "pregunta del journey", "qué dicen los datos", "¿se puede decidir con esto?"])

print(evidencia.to_string(index=False))
print(f"\npreguntas del journey que los datos contestan: "
      f"{(evidencia['¿se puede decidir con esto?'] == 'SÍ').sum()} de {len(evidencia)}")

## 4. Plantilla de customer journey map

Se llena **en equipo y en papel**, una columna por etapa. La regla de oro: cada casilla se llena con lo
que hace o siente **el cliente**, no con lo que hace la empresa. Si escribes «se registra la venta»,
esa frase va al blueprint, no aquí.

| | 1 · Descubrimiento | 2 · Consideración | 3 · Compra | 4 · Retención | 5 · Recomendación |
|---|---|---|---|---|---|
| **Qué hace el cliente** | | | | | |
| **Qué piensa** (frase textual, entre comillas) | | | | | |
| **Qué siente** (😀 😐 😟, una sola cara) | | | | | |
| **Punto de contacto** (dónde ocurre) | | | | | |
| **Punto de dolor** (qué le sale mal) | | | | | |
| **Dato que se genera aquí** | | | | | |
| **Dato que se pierde aquí** | | | | | |
| **Oportunidad** (qué cambiaría esto) | | | | | |

Tres reglas de llenado que separan un mapa útil de un dibujo bonito:

1. **La fila de emociones lleva una sola cara por etapa.** Si el equipo no se pone de acuerdo, la
   discusión es el ejercicio: significa que hay dos clientes distintos y hacen falta dos mapas.
2. **Las frases de «qué piensa» van entre comillas y en primera persona.** «No sé si me van a cobrar
   el envío» es utilizable; «incertidumbre sobre costos logísticos» no lo es.
3. **Las dos filas de datos se llenan al final**, cuando el mapa ya está completo, y con la hoja de
   evidencia de la sección 3 al lado. Ahí es donde el ejercicio deja de ser de diseño y pasa a ser de
   analítica.

## 5. Plantilla de service blueprint

El blueprint toma la misma línea de tiempo del journey y le añade lo que pasa hacia abajo, separado por
tres líneas horizontales que no se cruzan sin consecuencias.

| capa | qué va aquí | 1 · Descubrimiento | 2 · Consideración | 3 · Compra | 4 · Retención | 5 · Recomendación |
|---|---|---|---|---|---|---|
| **Acciones del cliente** | lo que hace la persona | | | | | |
| *— línea de interacción —* | | | | | | |
| **Evidencia física / interfaz** | lo que el cliente ve y toca | | | | | |
| **Personal de contacto** | lo que hace quien lo atiende | | | | | |
| *— línea de visibilidad —* | | | | | | |
| **Procesos de trastienda** | lo que ocurre y el cliente no ve | | | | | |
| *— línea de interacción interna —* | | | | | | |
| **Sistemas de soporte** | qué software sostiene cada paso | | | | | |
| **📌 Punto de captura de dato** | qué campo se graba y en qué tabla | | | | | |
| **⚠️ Decisión sin evidencia** | qué se decide aquí a ojo | | | | | |

Las dos últimas filas son las que convierten un blueprint de manual de servicio en un blueprint de
analítica, y son la entrega de la semana. Sobre el de Comercial Andina, las casillas ya se pueden
llenar con los números de la sección 3: en la columna de Compra va *«se graba `cliente_id`, y falta en
el 7,79 % de las líneas»*; en la de Consideración va una casilla de captura **vacía** y una decisión sin
evidencia que dice *«qué producto se pone a la entrada»*.

## 6. Cuánto vale identificar al cliente en el punto de venta

Aquí el ejercicio deja de ser descriptivo. Una mejora de proceso vale lo que produce menos lo que
cuesta, y las dos partes se estiman **con supuestos escritos**. Un supuesto declarado y discutible vale
mucho más que una cifra sin origen: el que revisa puede cambiarlo y ver qué pasa.

In [ ]:
ventas_l = ventas_limpias.merge(productos[["producto_id", "costo_unitario"]], on="producto_id",
                                how="left", validate="m:1")
ventas_l["monto"] = (ventas_l["cantidad"] * ventas_l["precio_unitario"] * (1 - ventas_l["descuento"]))
ventas_l["margen"] = ventas_l["monto"] - ventas_l["cantidad"] * ventas_l["costo_unitario"]
ANIOS = ((HOY.to_period("M") - ventas_l["fecha"].min().to_period("M")).n + 1) / 12

margen_cliente = (ventas_l.groupby("cliente_id")["margen"].sum() / ANIOS).rename("margen_anual")
margen_cliente = margen_cliente.to_frame().join(clientes.set_index("cliente_id")["tipo_cliente"])
print(f"periodo cubierto por la base: {ANIOS:.2f} años\n")
print("Margen anual que deja un cliente IDENTIFICADO, según su tipo:")
print(margen_cliente.groupby("tipo_cliente")["margen_anual"]
      .agg(clientes="size", media="mean", mediana="median").round(2).to_string())

tienda = ventas[ventas["canal"] == "Tienda"]
FACTURAS_ANONIMAS_ANIO = tienda.loc[tienda["sin_cliente"], "factura_id"].nunique() / ANIOS
VENTA_ANONIMA_ANIO = tienda.loc[tienda["sin_cliente"], "monto"].sum() / ANIOS
print(f"\nEn tienda física, cada año:")
print(f"  facturas con alguna línea sin identificar : {FACTURAS_ANONIMAS_ANIO:,.0f}")
print(f"  facturación que queda sin dueño           : {VENTA_ANONIMA_ANIO:,.2f}")

In [ ]:
# --- Supuestos declarados. Cámbialos y vuelve a correr: para eso están arriba y con nombre. ---
TASA_IDENTIFICACION = 0.60   # qué proporción de las compras anónimas logra identificar el cajero
MARGEN_MINORISTA = margen_cliente.query("tipo_cliente == 'Minorista'")["margen_anual"].mean()
MEJORA_POR_GESTION = 0.20    # cuánto más margen deja al año un cliente al que sí se le puede escribir
COSTO_CAPACITACION = 1200    # una vez: formar a los cajeros de las cinco tiendas
COSTO_INCENTIVO = 0.15       # por cada cliente identificado, incentivo al cajero
COSTO_SISTEMA_ANIO = 900     # ajuste del punto de venta y mantenimiento


def valor_de_identificar(tasa=TASA_IDENTIFICACION, mejora=MEJORA_POR_GESTION,
                         margen=MARGEN_MINORISTA, facturas=FACTURAS_ANONIMAS_ANIO):
    identificados = facturas * tasa
    beneficio = identificados * margen * mejora
    costo = identificados * COSTO_INCENTIVO + COSTO_SISTEMA_ANIO + COSTO_CAPACITACION
    return pd.Series({"clientes identificados al año": identificados,
                      "margen adicional": beneficio,
                      "costo anual": costo,
                      "valor neto anual": beneficio - costo})


resultado = valor_de_identificar()
print(f"margen anual medio de un cliente minorista identificado: {MARGEN_MINORISTA:,.2f}\n")
print(resultado.round(2).to_string())
print(f"\nventa de tienda que deja de ser anónima: "
      f"{VENTA_ANONIMA_ANIO * TASA_IDENTIFICACION:,.2f} al año "
      f"({TASA_IDENTIFICACION:.0%} de {VENTA_ANONIMA_ANIO:,.2f})")

📌 **El número es pequeño y hay que decirlo: 3 040,04 dólares de valor neto al año.** Identificar a
seis de cada diez compradores anónimos suma 917 clientes nuevos al radar comercial, y cada uno deja
28,77 dólares de margen al año, de los que la gestión captura un 20 % extra: 5 277,63 en total. Contra
2 237,59 dólares de costo, el proyecto queda en positivo por tres mil dólares al año. Para una empresa
que factura más de un millón, eso es ruido.

Con esa cifra sobre la mesa, la conversación cambia por completo: **no es un proyecto de sistemas, es un
proyecto de margen y su valor depende de tres supuestos que nadie ha medido.** Antes de defenderlo hay
que saber cuál de los tres lo hunde.

In [ ]:
filas = []
for tasa in [0.20, 0.40, 0.60, 0.80]:
    for mejora in [0.05, 0.10, 0.20, 0.35]:
        r = valor_de_identificar(tasa=tasa, mejora=mejora)
        filas.append((tasa, mejora, r["valor neto anual"]))
sens = (pd.DataFrame(filas, columns=["tasa de identificación", "mejora por gestión", "valor neto"])
        .pivot(index="tasa de identificación", columns="mejora por gestión", values="valor neto"))
print("Valor neto anual según los dos supuestos que más pesan:\n")
print(sens.round(0).to_string())
print(f"\ncombinaciones con valor negativo: {int((sens < 0).sum().sum())} de {sens.size}")

fig, ax = plt.subplots(figsize=(8.5, 3.6))
sns.heatmap(sens, annot=True, fmt=",.0f", center=0, cmap="RdYlGn", ax=ax, cbar=False)
ax.set_title("Con una mejora por gestión del 5 %, identificar al cliente pierde dinero en todos los casos")
ax.set_xlabel("mejora de margen que produce poder gestionar al cliente")
ax.set_ylabel("proporción de compras anónimas identificadas")
plt.tight_layout()
plt.show()

⚠️ **En siete de las dieciséis combinaciones el proyecto pierde dinero.** Y el supuesto que decide
no es el operativo —la tasa de identificación, que es en lo que todo el mundo se enfoca— sino el
comercial: **con una mejora por gestión del 5 % la columna entera es negativa, incluso identificando al
80 % de los compradores.** Identificar por identificar es un costo.

De ahí sale la recomendación honesta de esta semana: antes de cambiar el punto de venta, **medir el
supuesto de la mejora por gestión con la prueba A/B de la semana 9**. Es exactamente el experimento de
reactivación, y ya dio un resultado: la campaña casi duplicó la conversión de los minoristas, del 5,62 %
al 11,17 %. Ese es el número que hay que convertir en margen antes de firmar nada.

### 🌶️ Ejercicio 1 — Guiado

Repite la hoja de evidencia de la sección 3 **para una sola sucursal**. Elige la ciudad del negocio de
tu caso o, si no aplica, Manta, que es la más pequeña. Calcula para ella: qué porcentaje de sus líneas
no tiene cliente, cuántos de sus clientes son multicanal, cuántos llevan más de 180 días sin comprar y
cuál es la mediana de días de relación. Después escribe una frase por cifra: **qué casilla del blueprint
ilumina cada una.**

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: la ciudad de la venta está en sucursales.csv (sucursal_id → ciudad);
#          la ciudad del cliente está en clientes.csv y NO son la misma cosa
# Pista 2: ventas.query("ciudad == 'Manta'") y repite los tres cálculos de la sección 3
# Pista 3: compara cada porcentaje contra el global. Una diferencia de 2 puntos en una
#          sucursal pequeña puede ser ruido: mira también el número absoluto

### 🔥 Desafío

La etapa 2 del journey —la consideración— tiene cero datos. **Diseña el dato que falta.** Escribe la
ficha de captura de tres indicadores nuevos que Comercial Andina podría empezar a registrar mañana en
la tienda física, y para cada uno: qué mide exactamente, quién lo captura y en qué momento, cuánto
cuesta capturarlo, qué decisión habilita y **cuál es el número que hoy se decide a ojo y pasaría a
decidirse con evidencia**. Después ponle precio a uno de los tres con el mismo molde de la sección 6.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: los tres candidatos clásicos del retail físico son visitas a la tienda (contador en
#          la puerta), tasa de conversión visita→compra y motivo de no compra (pregunta del cajero)
# Pista 2: copia la función valor_de_identificar y cámbiale los supuestos. Lo que se califica
#          es que cada supuesto tenga nombre, valor y una línea de justificación
# Pista 3: un indicador que no cambia ninguna decisión no se captura, por barato que sea.
#          Escribe la decisión antes que el indicador

### 🎯 Reto en clase (15 min)

En equipos, con post-its y contra reloj. Cada equipo dibuja el journey map del proceso central de **otro**
equipo, usando solo lo que ese equipo contó en dos minutos, y lo devuelve. El equipo dueño marca en rojo
todo lo que está mal. Se puntúan dos cosas: cuántas etapas acertó el equipo visitante y —lo que de
verdad importa— **cuántos puntos de dolor detectó el visitante que el dueño no había visto**. El sesgo
de quien conoce el proceso por dentro es el enemigo del journey map, y esta es la forma barata de
medirlo.

In [ ]:
# TU CÓDIGO AQUÍ (esta celda es para la evidencia, no para el mapa: el mapa va en papel)
# Pista: mientras el otro equipo dibuja, tú prepara los tres números que vas a usar para
#        corregirlo. El molde es el de la sección 3:
#   (ventas
#    .groupby("<etapa o punto de contacto>")
#    .agg(lineas=("monto", "size"), sin_cliente=("sin_cliente", "mean"), dinero=("monto", "sum")))

## La trampa de hoy

⚠️ **Confundir tener un tablero con haberse transformado.** El tablero que nadie abre es un costo, no
una capacidad. Y se detecta con una pregunta que no es técnica: por cada indicador del tablero, ¿qué
decisión cambia según su valor y quién la toma?

Aquí lo medimos. Construimos el tablero real de Comercial Andina —seis indicadores, todos correctos,
todos calculables con los datos que hay— y al lado la lista de las seis decisiones del journey. Después
se cuentan los dos números.

In [ ]:
venta_online = compras[compras["canal"] == "Online"]["monto"].sum() / ANIOS
ticket = compras.groupby("factura_id")["monto"].sum().mean()

tablero = pd.DataFrame([
    ("Facturación anual", f"{ventas_l['monto'].sum() / ANIOS:,.2f}"),
    ("Margen anual", f"{ventas_l['margen'].sum() / ANIOS:,.2f}"),
    ("Ticket medio", f"{ticket:,.2f}"),
    ("Clientes activos", f"{compras['cliente_id'].nunique():,}"),
    ("Facturas al año", f"{compras['factura_id'].nunique() / ANIOS:,.0f}"),
    ("Ventas del canal en línea", f"{venta_online:,.2f}"),
], columns=["indicador del tablero", "valor"])

decisiones = pd.DataFrame([
    ("¿cuánto inventario pido de cada producto por tienda?", "ventas por producto y sucursal", "SÍ"),
    ("¿a qué clientes llamo esta semana?", "cliente identificado en cada compra", "NO"),
    ("¿qué producto pongo en la entrada de la tienda?", "visitas y recorrido en tienda", "NO"),
    ("¿por qué se fue este cliente?", "motivo de baja", "NO"),
    ("¿qué canal de captación funciona mejor?", "canal y fecha de alta completos", "PARCIAL"),
    ("¿cuándo vuelve a comprar un cliente?", "historial por cliente", "SÍ"),
], columns=["decisión del journey", "dato que necesita", "¿el dato existe?"])

print("LO QUE EL TABLERO MUESTRA")
print(tablero.to_string(index=False))
print("\nLO QUE EL NEGOCIO DECIDE")
print(decisiones.to_string(index=False))

listos = int((decisiones["¿el dato existe?"] == "SÍ").sum())
print(f"\nindicadores del tablero, todos correctos : {len(tablero)}")
print(f"decisiones del journey con datos detrás   : {listos} de {len(decisiones)}")
print(f"decisiones que hoy se toman a ciegas      : "
      f"{int((decisiones['¿el dato existe?'] == 'NO').sum())}")
print(f"\netapas del journey que el tablero cubre   : 1 de 5 (solo la compra)")

📌 **Seis indicadores impecables contra dos decisiones habilitadas.** El tablero de Comercial
Andina está bien construido, cuadra con la contabilidad y mide con precisión **una sola de las cinco
etapas** del journey: la compra. Las otras cuatro no tienen indicador porque no tienen dato, y no tienen
dato porque nadie dibujó nunca el proceso completo para darse cuenta.

Ese es el argumento de la semana entero: la transformación digital no empieza comprando un tablero,
empieza sabiendo **qué se decide, dónde y con qué evidencia**. Un tablero construido sin ese mapa
termina midiendo lo que es fácil de medir —que casi siempre es la transacción— y dejando fuera lo que
decide el negocio.

La regla práctica: **por cada indicador de un tablero, escribe al lado el nombre de la persona que
cambia algo cuando el número se mueve.** Los indicadores que quedan sin nombre se borran. En un tablero
real desaparece la mitad, y el que queda se usa.

## Entregable

Sube `lab_10_apellido.ipynb` y las dos plantillas llenas, con:

- El **customer journey map** del proceso central del negocio del caso, con las cinco etapas, la fila de
  emociones resuelta y las frases del cliente entre comillas.
- El **service blueprint** con las cuatro capas y las tres líneas, y sobre él las dos filas de
  analítica: **📌 punto de captura** y **⚠️ decisión sin evidencia**, con al menos tres decisiones
  marcadas con un círculo.
- La hoja de evidencia del negocio del caso: qué porcentaje de las ventas no se puede atribuir a un
  cliente, cuántos clientes cruzan de canal, cuánto dura la relación y qué parte del padrón está
  incompleta. Si su empresa no tiene alguno de esos datos, la respuesta correcta es *«no existe»*, y esa
  casilla vale igual que un número.
- La cuantificación de **una** de las tres decisiones sin evidencia, con el molde de la sección 6: los
  supuestos con nombre y valor, el resultado y la tabla de sensibilidad que muestre bajo qué supuestos
  el proyecto pierde dinero.
- Una fila nueva en la bitácora de prompts: le pediste al asistente el journey map de un cliente tipo y
  lo contrastaste con lo que dijo una persona real del negocio. Anota **las tres diferencias más
  grandes**: ahí está el valor de la semana.

## Para tu equipo

- El journey map se hace con alguien que haya trabajado en el proceso, no con lo que ustedes suponen que
  pasa. Media hora con la persona de la caja o del mostrador cambia más el mapa que tres horas de
  reunión entre analistas.
- Las decisiones marcadas con ⚠️ son el catálogo de problemas de la semana 11: de ahí salen los doce
  problemas que se van a clasificar por familia, línea base y valor anual. Cuantas más marquen esta
  semana, más fácil será la siguiente.
- Si al terminar el blueprint la fila de sistemas dice «Excel» en cuatro de las cinco columnas, no lo
  disimulen. Es el diagnóstico de madurez analítica más honesto que van a producir en el semestre, y es
  el punto de partida real de cualquier recomendación.